# Why Does CMM Improve Momentum?

This notebook runs paper-style diagnostic tests to explain why characteristic-managed/deep-learning weighted momentum (`CMM`) improves over standard momentum.

Core idea from the paper: all past return days are not equally informative. The neural network maps firm characteristics into a scalar `z_hat`, then uses `softmax(z_hat * daily_return)` to decide which days inside the past-return window receive more weight. The tests below check whether the improvement comes from stronger out-of-sample prediction, incremental information beyond standard momentum, risk exposure, and learned weight concentration.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

DATA_PATH = Path('../result/datasets/cmm_model_training_data.parquet')
PRED_PATH = Path('../result/models/cmm/cmm_predictions.parquet')
RETURN_COLS_PATH = Path('../result/datasets/cmm_return_window_columns.txt')
FEATURE_COLS_PATH = Path('../result/datasets/cmm_feature_columns.txt')
OUT_DIR = Path('../result/reports/cmm_explain')
OUT_DIR.mkdir(parents=True, exist_ok=True)

EVAL_SPLIT = 'test'

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)
plt.style.use('seaborn-v0_8-whitegrid')


## 1. Load Out-of-Sample Predictions and Build Baselines

`Standard Momentum` here uses the same CMM return window `ret_lag_252` through `ret_lag_22`, so this section isolates the contribution of learned weighting inside the same historical-return window. The production comparison notebook uses the same paper-style skip-one-month traditional momentum benchmark.

In [ ]:
return_cols = RETURN_COLS_PATH.read_text(encoding='utf-8').splitlines()
feature_cols = FEATURE_COLS_PATH.read_text(encoding='utf-8').splitlines()

pred = pd.read_parquet(PRED_PATH)
pred['signal_date'] = pd.to_datetime(pred['signal_date'])
pred = pred[pred['split'] == EVAL_SPLIT].copy()
if pred.duplicated(['stock_id', 'signal_date']).any():
    raise ValueError('Duplicate test predictions detected.')

base_cols = list(dict.fromkeys(['stock_id', 'signal_date', 'target_1m_ret', 'target_1m_ret_cs_z', 'ind1', 'z_tmv'] + return_cols + feature_cols))
base = pd.read_parquet(DATA_PATH, columns=base_cols)
base['signal_date'] = pd.to_datetime(base['signal_date'])
base['ind1'] = base['ind1'].fillna('Unknown')

df = pred[['stock_id', 'signal_date', 'target_1m_ret', 'target_1m_ret_cs_z', 'cmm_signal', 'cmm_signal_cs_z', 'z_hat', 'max_weight', 'fold']].merge(
    base, on=['stock_id', 'signal_date', 'target_1m_ret', 'target_1m_ret_cs_z'], how='inner'
)

# Same-window standard momentum: equal-weighted log-return sum over the CMM formation window.
df['same_window_mom'] = df[return_cols].sum(axis=1)
df['same_window_mom_cs_z'] = df.groupby('signal_date')['same_window_mom'].transform(lambda s: (s - s.mean()) / s.std(ddof=0))

print(df.shape)
print(df['signal_date'].min(), df['signal_date'].max(), df['signal_date'].nunique())
df[['stock_id','signal_date','cmm_signal_cs_z','same_window_mom_cs_z','z_hat','max_weight','target_1m_ret']].head()


## 2. Out-of-Sample IC and Rank IC

If CMM truly improves momentum, its month-by-month correlation with next-month returns should be stronger and more stable than equal-weighted momentum.

In [ ]:
def monthly_ic(frame, factor_col, ret_col='target_1m_ret'):
    rows = []
    for date, g in frame.groupby('signal_date', sort=True):
        x = g[factor_col]
        y = g[ret_col]
        valid = x.notna() & y.notna()
        if valid.sum() < 50:
            continue
        rows.append({
            'signal_date': date,
            'factor': factor_col,
            'pearson_ic': x[valid].corr(y[valid], method='pearson'),
            'spearman_rank_ic': x[valid].corr(y[valid], method='spearman'),
            'n': int(valid.sum()),
        })
    return pd.DataFrame(rows)

ic = pd.concat([
    monthly_ic(df, 'cmm_signal_cs_z'),
    monthly_ic(df, 'same_window_mom_cs_z'),
], ignore_index=True)

def summarize_ic(ic_df):
    rows = []
    for factor, g in ic_df.groupby('factor'):
        for col in ['pearson_ic', 'spearman_rank_ic']:
            s = g[col].dropna()
            rows.append({
                'factor': factor,
                'metric': col,
                'months': len(s),
                'mean': s.mean(),
                'std': s.std(ddof=1),
                't_stat': s.mean() / (s.std(ddof=1) / np.sqrt(len(s))),
                'positive_rate': (s > 0).mean(),
            })
    return pd.DataFrame(rows)

ic_summary = summarize_ic(ic)
ic.to_csv(OUT_DIR / 'monthly_ic.csv', index=False)
ic_summary.to_csv(OUT_DIR / 'ic_summary.csv', index=False)
ic_summary


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
for factor, label in [('cmm_signal_cs_z', 'CMM'), ('same_window_mom_cs_z', 'Equal-weight momentum')]:
    g = ic[(ic['factor'] == factor)].sort_values('signal_date')
    ax.plot(g['signal_date'], g['spearman_rank_ic'].rolling(6, min_periods=1).mean(), linewidth=2, label=label)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('6-Month Rolling Rank IC')
ax.set_xlabel('Signal date')
ax.set_ylabel('Rank IC')
ax.legend()
fig.tight_layout()
ic_path = OUT_DIR / 'rolling_rank_ic.png'
fig.savefig(ic_path, dpi=160, bbox_inches='tight')
plt.close(fig)
ic_path


## 3. Incremental Information Beyond Equal-Weighted Momentum

This is the most direct diagnostic. Each month, run a cross-sectional regression:

```text
next_month_return = a + b1 * CMM + b2 * equal_weight_momentum + error
```

If `b1` stays positive and significant while `b2` is controlled for, CMM is adding information beyond ordinary momentum.

In [ ]:
def monthly_cs_regression(frame, y_col, x_cols):
    rows = []
    for date, g in frame.groupby('signal_date', sort=True):
        use = g[[y_col] + x_cols].replace([np.inf, -np.inf], np.nan).dropna()
        if len(use) < len(x_cols) + 30:
            continue
        y = use[y_col].to_numpy(dtype=float)
        x = use[x_cols].to_numpy(dtype=float)
        x = np.column_stack([np.ones(len(use)), x])
        beta, *_ = np.linalg.lstsq(x, y, rcond=None)
        row = {'signal_date': date, 'n': len(use), 'intercept': beta[0]}
        for name, value in zip(x_cols, beta[1:]):
            row[name] = value
        rows.append(row)
    return pd.DataFrame(rows)

reg = monthly_cs_regression(df, 'target_1m_ret_cs_z', ['cmm_signal_cs_z', 'same_window_mom_cs_z'])
reg_summary = []
for col in ['cmm_signal_cs_z', 'same_window_mom_cs_z']:
    s = reg[col].dropna()
    reg_summary.append({
        'coefficient': col,
        'months': len(s),
        'mean_slope': s.mean(),
        't_stat': s.mean() / (s.std(ddof=1) / np.sqrt(len(s))),
        'positive_rate': (s > 0).mean(),
    })
reg_summary = pd.DataFrame(reg_summary)
reg.to_csv(OUT_DIR / 'monthly_incremental_regression.csv', index=False)
reg_summary.to_csv(OUT_DIR / 'incremental_regression_summary.csv', index=False)
reg_summary


## 4. Is CMM Just A Rescaled Momentum Signal?

If CMM were merely standard momentum in disguise, the cross-sectional correlation between CMM and equal-weighted momentum would be close to 1 and CMM would not add much after controlling for momentum.

In [ ]:
corr_rows = []
for date, g in df.groupby('signal_date', sort=True):
    corr_rows.append({
        'signal_date': date,
        'cmm_vs_same_window_mom_rank_corr': g['cmm_signal_cs_z'].corr(g['same_window_mom_cs_z'], method='spearman'),
        'cmm_vs_same_window_mom_pearson_corr': g['cmm_signal_cs_z'].corr(g['same_window_mom_cs_z'], method='pearson'),
    })
factor_corr = pd.DataFrame(corr_rows)
factor_corr_summary = factor_corr.drop(columns='signal_date').agg(['mean','std','min','max']).T
factor_corr.to_csv(OUT_DIR / 'monthly_factor_correlations.csv', index=False)
factor_corr_summary.to_csv(OUT_DIR / 'factor_correlation_summary.csv')
factor_corr_summary


## 5. What Did The Network Learn To Weight?

Given the paper's mechanism, the learned scalar `z_hat` controls how concentrated the softmax weights are and whether positive or negative historical-return days receive more emphasis. Here we reconstruct the 231-day weights for the test set.

In [ ]:
ret_mat = df[return_cols].to_numpy(dtype=np.float32)
z_hat = df['z_hat'].to_numpy(dtype=np.float32).reshape(-1, 1)
scores = z_hat * ret_mat
scores = scores - scores.max(axis=1, keepdims=True)
w = np.exp(scores).astype(np.float32)
w = w / w.sum(axis=1, keepdims=True)

ret_mean = ret_mat.mean(axis=1)
weighted_ret = (w * ret_mat).sum(axis=1)
positive_weight_share = (w * (ret_mat > 0)).sum(axis=1)
negative_weight_share = (w * (ret_mat < 0)).sum(axis=1)
effective_days = 1.0 / np.square(w).sum(axis=1)

df['equal_weight_window_mean'] = ret_mean
df['learned_weight_window_return'] = weighted_ret
df['weighting_gain_vs_equal_mean'] = weighted_ret - ret_mean
df['positive_weight_share'] = positive_weight_share
df['negative_weight_share'] = negative_weight_share
df['effective_weighted_days'] = effective_days

weight_summary = df[['z_hat','max_weight','positive_weight_share','negative_weight_share','effective_weighted_days','weighting_gain_vs_equal_mean']].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).T
weight_summary.to_csv(OUT_DIR / 'weight_mechanism_summary.csv')
weight_summary


In [ ]:
# Average weight profile over event time t-252 ... t-22 for high/low z_hat groups.
df['z_hat_decile'] = df.groupby('signal_date')['z_hat'].transform(lambda s: pd.qcut(s.rank(method='first'), 10, labels=False) + 1)
low_idx = df['z_hat_decile'].eq(1).to_numpy()
high_idx = df['z_hat_decile'].eq(10).to_numpy()
profile = pd.DataFrame({
    'lag': [int(c.replace('ret_lag_', '')) for c in return_cols],
    'avg_weight_low_z_hat': w[low_idx].mean(axis=0),
    'avg_weight_high_z_hat': w[high_idx].mean(axis=0),
})
profile.to_csv(OUT_DIR / 'weight_profile_by_z_hat_decile.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.plot(profile['lag'], profile['avg_weight_low_z_hat'], label='Low z_hat decile', linewidth=2)
ax.plot(profile['lag'], profile['avg_weight_high_z_hat'], label='High z_hat decile', linewidth=2)
ax.invert_xaxis()
ax.set_title('Average Learned Daily Weights by z_hat Decile')
ax.set_xlabel('Return lag, trading days before signal')
ax.set_ylabel('Average softmax weight')
ax.legend()
fig.tight_layout()
weight_profile_path = OUT_DIR / 'weight_profile_by_z_hat_decile.png'
fig.savefig(weight_profile_path, dpi=160, bbox_inches='tight')
plt.close(fig)
weight_profile_path


## 6. Which Characteristics Move `z_hat`?

The paper's interpretation is characteristic-managed momentum: the same past return path can be weighted differently for different firms. This section checks which firm characteristics are most associated with the learned scalar.

In [ ]:
feature_corr_rows = []
for col in feature_cols:
    vals = []
    for date, g in df.groupby('signal_date', sort=True):
        c = g[col].corr(g['z_hat'], method='spearman')
        if pd.notna(c):
            vals.append(c)
    if vals:
        s = pd.Series(vals)
        feature_corr_rows.append({
            'feature': col,
            'mean_rank_corr_with_z_hat': s.mean(),
            'abs_mean_rank_corr': abs(s.mean()),
            't_stat': s.mean() / (s.std(ddof=1) / np.sqrt(len(s))) if s.std(ddof=1) > 0 else np.nan,
            'positive_rate': (s > 0).mean(),
        })
feature_corr = pd.DataFrame(feature_corr_rows).sort_values('abs_mean_rank_corr', ascending=False)
feature_corr.to_csv(OUT_DIR / 'feature_correlations_with_z_hat.csv', index=False)
feature_corr.head(20)


In [ ]:
top = feature_corr.head(15).sort_values('mean_rank_corr_with_z_hat')
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top['feature'], top['mean_rank_corr_with_z_hat'])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top Characteristics Associated With z_hat')
ax.set_xlabel('Mean monthly Spearman correlation')
fig.tight_layout()
feature_path = OUT_DIR / 'top_features_with_z_hat.png'
fig.savefig(feature_path, dpi=160, bbox_inches='tight')
plt.close(fig)
feature_path


## 7. First-Order Mechanism Check

Because the learned softmax weights are usually close to equal weights, the improvement may come from the first-order interaction implied by the paper's formula rather than from putting all weight on a few days. For small `z_hat * r`,

```text
softmax(z_hat * r) weighted return ≈ mean(r) + z_hat * variance(r)
```

This test checks whether CMM is mainly a characteristic-managed transformation of the return-window distribution.

In [ ]:
df['window_var'] = df[return_cols].var(axis=1, ddof=0)
df['first_order_cmm_approx'] = df['equal_weight_window_mean'] + df['z_hat'] * df['window_var']

approx_rows = []
for date, g in df.groupby('signal_date', sort=True):
    approx_rows.append({
        'signal_date': date,
        'cmm_vs_first_order_corr': g['cmm_signal'].corr(g['first_order_cmm_approx'], method='pearson'),
        'cmm_vs_equal_mean_corr': g['cmm_signal'].corr(g['equal_weight_window_mean'], method='pearson'),
        'first_order_rank_ic': g['first_order_cmm_approx'].corr(g['target_1m_ret'], method='spearman'),
        'z_hat_x_var_rank_ic': (g['z_hat'] * g['window_var']).corr(g['target_1m_ret'], method='spearman'),
    })
approx = pd.DataFrame(approx_rows)
approx_summary = approx.drop(columns='signal_date').agg(['mean','std','min','max']).T
approx.to_csv(OUT_DIR / 'first_order_mechanism_monthly.csv', index=False)
approx_summary.to_csv(OUT_DIR / 'first_order_mechanism_summary.csv')
approx_summary


## 8. Summary Table

This table condenses the explanation into a few diagnostics: prediction strength, incremental information, relationship with equal-weighted momentum, and learned-weight behavior.

In [ ]:
summary_rows = []
rank_ic_summary = ic_summary[ic_summary['metric'] == 'spearman_rank_ic'].set_index('factor')
summary_rows.append({'diagnostic':'CMM mean rank IC', 'value': rank_ic_summary.loc['cmm_signal_cs_z','mean']})
summary_rows.append({'diagnostic':'Equal-weight momentum mean rank IC', 'value': rank_ic_summary.loc['same_window_mom_cs_z','mean']})
summary_rows.append({'diagnostic':'CMM incremental regression slope t-stat', 'value': reg_summary.set_index('coefficient').loc['cmm_signal_cs_z','t_stat']})
summary_rows.append({'diagnostic':'Momentum incremental regression slope t-stat', 'value': reg_summary.set_index('coefficient').loc['same_window_mom_cs_z','t_stat']})
summary_rows.append({'diagnostic':'Mean rank corr: CMM vs equal-weight momentum', 'value': factor_corr['cmm_vs_same_window_mom_rank_corr'].mean()})
summary_rows.append({'diagnostic':'Median effective weighted days', 'value': df['effective_weighted_days'].median()})
summary_rows.append({'diagnostic':'Median max daily weight', 'value': df['max_weight'].median()})
summary_rows.append({'diagnostic':'Mean positive-day weight share', 'value': df['positive_weight_share'].mean()})
summary_rows.append({'diagnostic':'Mean corr: CMM vs first-order approximation', 'value': approx['cmm_vs_first_order_corr'].mean()})
summary_rows.append({'diagnostic':'Mean rank IC: first-order approximation', 'value': approx['first_order_rank_ic'].mean()})
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT_DIR / 'explanation_summary.csv', index=False)
summary
